# Improve AE champion update 280

Use this notebook after the main training run has produced `ae_actor_update_0280.pt`. It is designed to avoid destroying the champion: evaluate first, try checkpoint soup, then optionally run a short KL-anchored fine-tune branch.


In [ ]:
from pathlib import Path
import sys

CWD = Path.cwd().resolve()
if CWD.name == 'training':
    REPO = CWD.parents[1]
elif CWD.name == 'ae':
    REPO = CWD.parent
else:
    REPO = CWD

sys.path.insert(0, str(REPO / 'ae' / 'training'))
sys.path.insert(0, str(REPO / 'ae' / 'src'))
sys.path.insert(0, str(REPO / 'til-26-ae'))

print('Repo:', REPO)


## Import helpers

The heavy helper functions live in `improve_280_tools.py` so this notebook stays readable.


In [ ]:
# Uncomment in a fresh environment if needed.
# %pip install -r {REPO / 'ae' / 'training' / 'requirements.txt'}
# %pip install -e {REPO / 'til-26-ae'}


In [ ]:
import torch
import improve_280_tools as t

torch.set_num_threads(1)
t.set_seed(280)
print('Device:', t.DEVICE)
print('Artifacts:', t.AE_ARTIFACTS)
for name, path in t.CKPTS.items():
    print(name, path.exists(), path.name)


## Protect raw update 280

Run this first. It restores raw 280 to deployment position and makes a named backup.


In [ ]:
t.protect(t.CKPTS['280'], 'ae_actor_champion_0280.pt')
t.protect(t.CKPTS['280'], 'ae_actor.pt')


## Larger evaluation of strong checkpoints

Use 50 episodes as a minimum. Increase to 100 if there is time. The score shown is `mean - 0.25 * std`.


In [ ]:
baseline_rows = t.eval_table(t.CKPTS, episodes=50, seed=26)
baseline_rows


## Try checkpoint soup

This creates a few weight-averaged candidates. In RL, soup can break the policy, so evaluate before promoting anything.


In [ ]:
soups = t.make_default_soups()
soups


In [ ]:
soup_rows = t.eval_table(soups, episodes=50, seed=2600) if soups else []
soup_rows


## Optional: KL-anchored fine-tune from 280

This starts from update 280, trains with very low LR, samples checkpoint-pool opponents, and adds a KL penalty to keep the new policy close to frozen 280. Only promote it if a later large eval beats raw 280.


In [ ]:
ft_cfg = t.FTConfig(
    updates=40,
    episodes_per_update=16,
    lr=2e-5,
    entropy_coef_start=0.006,
    entropy_coef_end=0.003,
    kl_anchor_beta=0.03,
    eval_every=2,
    eval_episodes=30,
    stop_entropy_below=0.55,
    stop_mean_below=450.0,
    stop_bad_evals=2,
)
ft_cfg


In [ ]:
# Run only after baseline/soup evals. This may take a while.
# best_ft_path, ft_logs = t.fine_tune_from_280(cfg=ft_cfg, seed=280)


## Evaluate fine-tune candidate

Uncomment after running the fine-tune cell. Compare against raw 280 using a fresh seed.


In [ ]:
# candidates = {
#     'raw_280': t.CKPTS['280'],
#     'ft_best': t.AE_ARTIFACTS / 'ae_actor_ft_best_from_0280.pt',
# }
# t.eval_table(candidates, episodes=50, seed=8800)


## Promote final winner

Only run the line matching the candidate that wins a larger eval. Safer default is raw 280.


In [ ]:
# Safer default: restore raw 280.
# t.protect(t.CKPTS['280'], 'ae_actor.pt')

# If a soup wins:
# t.protect(t.AE_ARTIFACTS / 'ae_actor_soup_080_280_020_285.pt', 'ae_actor.pt')

# If fine-tune wins:
# t.protect(t.AE_ARTIFACTS / 'ae_actor_ft_best_from_0280.pt', 'ae_actor.pt')


## Build after promotion

```bash
cd ae
docker build -t TEAM_ID-ae:finals .
```
